# Module 5 Homework: Data Platforms with Bruin

Based on [Module 5 Homework](https://github.com/DataTalksClub/data-engineering-zoomcamp/blob/main/cohorts/2026/05-data-platforms/homework.md).

This notebook contains the questions and answers. Setup: install Bruin CLI, initialize the zoomcamp template, configure `.bruin.yml` with DuckDB (or BigQuery — see STEP_BY_STEP.md), and follow the [module README](../README.md) tutorial.

**Run this notebook:**

1. `conda activate dataTalks` (optional), then open in Jupyter.
2. Run the **Setup** cell below first (sets paths and checks Bruin).
3. Run each part in order; code cells run Bruin CLI or show project files so you can verify answers.
4. See `STEP_BY_STEP.md` for full setup (Bruin CLI, `bruin init zoomcamp`, `.bruin.yml`).

In [9]:
# Setup: paths and Bruin check (run this first)
from pathlib import Path
import subprocess
import os

# Pipeline dir: from repo root, or from 05-data-platforms, or from HW5
cwd = Path.cwd()
if (cwd / "05-data-platforms" / "my-pipeline").exists():
    PIPELINE_DIR = cwd / "05-data-platforms" / "my-pipeline"
elif (cwd / "my-pipeline").exists():
    PIPELINE_DIR = cwd / "my-pipeline"
elif (cwd / ".." / "my-pipeline").resolve().exists():
    PIPELINE_DIR = (cwd / ".." / "my-pipeline").resolve()
else:
    PIPELINE_DIR = cwd / "my-pipeline"  # placeholder

def run_bruin(*args, cwd=None):
    """Run bruin CLI; returns (success, output)."""
    env = os.environ.copy()
    env["PATH"] = os.environ.get("PATH", "") + os.pathsep + os.path.expanduser("~/.local/bin")
    try:
        r = subprocess.run(
            ["bruin"] + list(args),
            capture_output=True,
            text=True,
            timeout=30,
            cwd=cwd or str(PIPELINE_DIR),
            env=env,
        )
        return r.returncode == 0, (r.stdout or "") + (r.stderr or "")
    except FileNotFoundError:
        return False, "bruin not found. Install: curl -LsSf https://getbruin.com/install/cli | sh"
    except Exception as e:
        return False, str(e)

# Check Bruin
ok, out = run_bruin("--version", cwd=os.getcwd())
print("Bruin:", out.strip() if ok else out)
print("Pipeline dir:", PIPELINE_DIR, "(exists:" + str(PIPELINE_DIR.exists()) + ")")

Bruin: Current: v0.11.478 (8210907696cb0160a6c14dcc38217e18fbdb1f88)
Latest: v0.11.478
Pipeline dir: /Users/sahand/Desktop/DataTalks/DataEngineering2025/data-engineering-zoomcamp/05-data-platforms/my-pipeline (exists:True)


### Question 1. Bruin Pipeline Structure

In a Bruin project, what are the required files/directories?

- `bruin.yml` and `assets/`
- `.bruin.yml` and `pipeline.yml` (assets can be anywhere)
- `.bruin.yml` and `pipeline/` with `pipeline.yml` and `assets/`
- `pipeline.yml` and `assets/` only

**Answer:** `.bruin.yml` and `pipeline/` with `pipeline.yml` and `assets/`

A Bruin project has a root `.bruin.yml` (environments, connections) and each pipeline lives in its own folder (e.g. `pipeline/` or `pipelines/nyc-taxi/`) containing `pipeline.yml` and an `assets/` directory.

In [10]:
# Part 1: Show required files/directories in the project
if PIPELINE_DIR.exists():
    for p in sorted(PIPELINE_DIR.iterdir()):
        name = p.name
        if name.startswith("."):
            name = name  # .bruin.yml
        print(f"  {p.name}/" if p.is_dir() else f"  {p.name}")
    if (PIPELINE_DIR / "pipeline").exists():
        print("  pipeline/")
        for q in sorted((PIPELINE_DIR / "pipeline").iterdir()):
            print(f"    {q.name}/" if q.is_dir() else f"    {q.name}")
else:
    print("Pipeline dir not found. Run from repo root or 05-data-platforms/HW5 after creating my-pipeline.")

  .bruin.yml
  .gitignore
  README.md
  duckdb.db
  pipeline/
  pipeline/
    assets/
    pipeline.yml


### Question 2. Materialization Strategies

You're building a pipeline that processes NYC taxi data organized by month based on `pickup_datetime`. Which incremental strategy is best for processing a specific interval period by **deleting and inserting data for that time period**?

- `append` - always add new rows
- `replace` - truncate and rebuild entirely
- `time_interval` - incremental based on a time column
- `view` - create a virtual table only

**Answer:** `time_interval`

For a specific interval (e.g. one month), deleting and re-inserting data for that period is the behavior of a **time-interval** incremental strategy: process only the given window and replace that slice.

In [11]:
# Part 2: Show materialization strategy (time_interval) in our staging asset
staging_sql = PIPELINE_DIR / "pipeline" / "assets" / "staging" / "trips.sql"
if staging_sql.exists():
    text = staging_sql.read_text()
    for line in text.splitlines():
        if "strategy:" in line or "time_interval" in line:
            print(line.strip())
else:
    print("staging/trips.sql not found; strategy for interval delete/insert is time_interval")

strategy: time_interval
-- When using `time_interval` strategy, Bruin:


### Question 3. Pipeline Variables

You have the following variable defined in `pipeline.yml`:

```yaml
variables:
  taxi_types:
    type: array
    items:
      type: string
    default: ["yellow", "green"]
```

How do you override this when running the pipeline to only process yellow taxis?

- `bruin run --taxi-types yellow`
- `bruin run --var taxi_types=yellow`
- `bruin run --var 'taxi_types=["yellow"]'`
- `bruin run --set taxi_types=["yellow"]`

**Answer:** `bruin run --var 'taxi_types=["yellow"]'`

Custom variables are overridden at runtime with `--var KEY=VALUE`. For an array type, the value is JSON, so you pass the array as a string: `--var 'taxi_types=["yellow"]'`.

In [12]:
# Part 3: Override variable — show --var in bruin run --help
ok, out = run_bruin("run", "--help", cwd=str(PIPELINE_DIR))
for line in out.splitlines():
    if "--var" in line:
        print(line)
print("\nExample: bruin run ./pipeline --config-file .bruin.yml --var 'taxi_types=[\"yellow\"]'")

   --var string [ --var string ]                  override pipeline variables with custom values [$BRUIN_VARS]

Example: bruin run ./pipeline --config-file .bruin.yml --var 'taxi_types=["yellow"]'


### Question 4. Running with Dependencies

You've modified the `ingestion/trips.py` asset and want to run it plus **all downstream assets**. Which command should you use?

- `bruin run ingestion.trips --all`
- `bruin run ingestion/trips.py --downstream`
- `bruin run pipeline/trips.py --recursive`
- `bruin run --select ingestion.trips+`

**Answer:** `bruin run ingestion/trips.py --downstream`

The `--downstream` flag runs the given asset and all assets that depend on it. You target the asset (by path or name) and add `--downstream` so everything that reads from it is re-run.

### Question 5. Quality Checks

You want to ensure the `pickup_datetime` column in your trips table **never has NULL values**. Which quality check should you add to your asset definition?

- `name: unique`
- `name: not_null`
- `name: positive`
- `name: accepted_values, value: [not_null]`

In [13]:
# Part 4: Run asset + downstream — run lineage to see downstream of ingestion.trips
asset_path = PIPELINE_DIR / "pipeline" / "assets" / "ingestion" / "trips.py"
if asset_path.exists():
    ok, out = run_bruin("lineage", str(asset_path.relative_to(PIPELINE_DIR)))
    print(out if out else "Run from terminal: bruin run ./pipeline/assets/ingestion/trips.py --downstream --config-file .bruin.yml")
else:
    print("ingestion/trips.py not found.")


Lineage: 'ingestion.trips'

Upstream Dependencies
Asset has no upstream dependencies.


Downstream Dependencies
- staging.trips (assets/staging/trips.sql)

Total: 1



**Answer:** `name: not_null`

A check that a column has no NULLs is the **not_null** quality check.

### Question 6. Lineage and Dependencies

After building your pipeline, you want to **visualize the dependency graph** between assets. Which Bruin command should you use?

- `bruin graph`
- `bruin dependencies`
- `bruin lineage`
- `bruin show`

**Answer:** `bruin lineage`

`bruin lineage` shows the dependency graph (upstream and downstream relationships) between assets.

In [14]:
# Part 5: Quality check for no NULLs — show not_null in our staging asset
staging_sql = PIPELINE_DIR / "pipeline" / "assets" / "staging" / "trips.sql"
if staging_sql.exists():
    text = staging_sql.read_text()
    for line in text.splitlines():
        if "not_null" in line:
            print(line)
else:
    print("Answer: name: not_null")

      - name: not_null


### Question 7. First-Time Run

You're running a Bruin pipeline for the **first time** on a new DuckDB database. What flag should you use to ensure tables are created from scratch?

- `--create`
- `--init`
- `--full-refresh`
- `--truncate`

**Answer:** `--full-refresh`

`--full-refresh` drops and recreates tables (overrides incremental materialization), so on a fresh database it ensures everything is built from scratch.

---

## Submitting the solutions

Form: <https://courses.datatalks.club/de-zoomcamp-2026/homework/hw5>

In [15]:
# Part 6: Visualize dependency graph — run bruin lineage
asset_path = PIPELINE_DIR / "pipeline" / "assets" / "ingestion" / "trips.py"
if asset_path.exists():
    ok, out = run_bruin("lineage", str(asset_path.relative_to(PIPELINE_DIR)))
    print(out)
else:
    print("Run in terminal: bruin lineage ./pipeline/assets/ingestion/trips.py")


Lineage: 'ingestion.trips'

Upstream Dependencies
Asset has no upstream dependencies.


Downstream Dependencies
- staging.trips (assets/staging/trips.sql)

Total: 1



In [16]:
# Part 7: First-time run — show --full-refresh in bruin run --help
ok, out = run_bruin("run", "--help", cwd=str(PIPELINE_DIR))
for line in out.splitlines():
    if "full-refresh" in line or "full_refresh" in line:
        print(line)
print("\nExample: bruin run ./pipeline --config-file .bruin.yml --full-refresh")

   --full-refresh, -r                             truncate the table before running (default: false) [$BRUIN_FULL_REFRESH]

Example: bruin run ./pipeline --config-file .bruin.yml --full-refresh
